In [1]:
import modules.data as d
import modules.model2 as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn
import torch.nn.functional as F

###

from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from pathlib import Path
from typing import Literal, Callable

In [ ]:
# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# get data
data = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    max_subset=120,
)

---

In [ ]:
data_module = t.DataModule(
    X=data.X,
    y=data.y,
    generator=generator,
    # batch_size=16,
    # val_size=0.15,
    # test_size=0.15
)

In [ ]:
class MultiTrainingModule():
    def __init__(self, model_class, model_kwargs, training_class, training_kwargs, num_trials:int, comment = None):
        # get dir
        self.folder = self._get_dir(comment)

        # init trackers
        self.trial_metrics = {}
        self.test_metrics = {}

        # trial loop
        for trial in tqdm(range(num_trials)):

            # define model
            model = model_class(**model_kwargs)

            # define, run experiment
            training_module = training_class(model=model, **training_kwargs)

            # save trial results
            self.trial_metrics[trial] = training_module.trial_metrics
            self.train_results = self._trial_to_csv(self.trial_metrics, 'train')
            self.val_results = self._trial_to_csv(self.trial_metrics, 'val')

            # save test results
            self.test_metrics[trial] = training_module.test_metrics
            self.test_results = self._test_to_csv(self.test_metrics)

            # save model
            state_dict = model.state_dict()
            torch.save(state_dict, self.folder / f'model_trial_{trial}.pth')

    def _get_dir(self, comment):
        # get date, time
        date = datetime.now().strftime("%Y-%m-%d")
        time = datetime.now().strftime("%Hh%Mm%Ss").lower()

        # get dir name
        dir_name = f'{date}_{time}'

        if (comment != None) & (type(comment) == str): 
            dir_name = dir_name + f'_{comment}' # append comment if applicable

        # create folder
        folder = Path(f'./output/{dir_name}')
        folder.mkdir(parents=True, exist_ok=True)

        return folder

    def _trial_to_csv(self, metrics_dict, col:str):
        # trial dict to dataframe
        results = pd.DataFrame(
            [
                {'trial':trial, 'epoch':epoch, **epoch_metrics[col]} # col: 'train' or 'val'
                for trial, trial_metrics in metrics_dict.items() # 
                for epoch, epoch_metrics in trial_metrics.items()
            ]
        )
        
        # write to csv
        results.to_csv(self.folder / f'{col}.csv')

        return results

    def _test_to_csv(self, metrics_dict, col:str='test'):
        # test dict to dataframe
        results = pd.DataFrame(
            [
                {'trial':trial, **metrics}
                for trial, metrics in metrics_dict.items()
            ]
        )

        # write to csv
        results.to_csv(self.folder / f'{col}.csv')

        return results